# 02. 아티스트 소통과 팬 리텐션 (H-01, H-02 검증)

> ⚠️ **가상 데이터 프로젝트**: 이 노트북의 모든 수치는 `FANLOG` 포트폴리오 프로젝트를 위해 생성한 가상 데이터를 대상으로 한다. 실제 팬덤에 대한 사실이 아니라 "가정한 시나리오의 탐지 결과"로 읽어야 한다(PRD 13.4절).

## 검증할 가설

- **H-01**: 아티스트 소통 활동일이 많은 주에는 팬 활동(WAU)과 참여율이 높게 나타날 것이다. (PRD 7.1절)
- **H-02**: 연속 소통 공백이 길수록 14일 이탈 위험 및 30일 휴면 비율이 높게 나타날 것이다. (PRD 7.1절)

## 지켜야 할 원칙 (PRD 14.4절, 14.6절)

- 상관관계를 인과관계로 표현하지 않는다.
- 모든 비율에 분자·분모(표본 수 n)를 함께 표시한다.
- 작은 표본의 결과를 확정적인 차이로 말하지 않는다.
- 분모가 0이면 `0%`가 아니라 `계산 불가`로 처리한다.
- 상관계수 크기 해석(약함/중간/강함)에 사용하는 기준 자체가 논쟁적일 수 있음을 `docs/decisions_log.md`의 "11. 검토 필요 항목"에 남긴다.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, norm

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.db import get_engine  # noqa: E402

engine = get_engine()
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 140)
print("DB 연결 완료:", engine.url.database)


## 상관계수 크기 해석 기준 (사전 고지)

이 노트북에서 Spearman 상관계수 `|r|`을 다음 기준으로 "약함/중간/강함"으로 표시한다.

| `|r|` 구간 | 표현 |
| --- | --- |
| `< 0.3` | 약함 |
| `0.3 ~ 0.5` | 중간 |
| `> 0.5` | 강함 |

이 구간은 심리학·사회과학 연구에서 흔히 인용되는 경험적 기준(Cohen 1988 계열) 중 하나를 채택한 것이며, 분야·맥락에 따라 다른 기준을 쓰기도 한다. **이 기준 자체가 유일한 정답은 아니라는 점**을 감안해서 읽어야 한다(`docs/decisions_log.md` 11절에 REVIEW NEEDED로 기록).

**신뢰구간 추가(`docs/decisions_log.md` 11.1절 보완)**: 점추정치(`r`)만 보면 표본이 작을 때의 불확실성이 가려질 수 있어, 모든 상관계수 결과표에 **Fisher z 변환 기반 95% 신뢰구간**(`ci_low`~`ci_high`)을 함께 표시한다. `n<15`인 경우 `note` 열에 "표본이 작아 신뢰구간이 넓음"을 자동으로 붙인다.


In [ ]:
def interpret_r(r: float) -> str:
    a = abs(r)
    if a < 0.3:
        return "약함"
    elif a <= 0.5:
        return "중간"
    else:
        return "강함"


def fisher_z_ci(r: float, n: int, n_control_vars: int = 0, confidence: float = 0.95) -> tuple:
    """Fisher z 변환 기반 상관계수 신뢰구간. n_control_vars는 편상관일 때 통제 변수 개수
    (자유도가 n_control_vars만큼 더 줄어든다). 표본이 부족하거나 |r|=1이면 계산하지 않는다."""
    dof = n - 3 - n_control_vars
    if dof <= 0 or pd.isna(r) or abs(r) >= 1:
        return (np.nan, np.nan)
    z = np.arctanh(r)
    se = 1 / np.sqrt(dof)
    z_crit = norm.ppf(1 - (1 - confidence) / 2)
    lo, hi = np.tanh(z - z_crit * se), np.tanh(z + z_crit * se)
    return (round(lo, 3), round(hi, 3))


def small_sample_note(n: int, threshold: int = 15) -> str:
    return "표본이 작아 신뢰구간이 넓음" if n < threshold else ""


def spearman_report(x: pd.Series, y: pd.Series) -> dict:
    """짝지어진 값 중 결측이 없는 것만 사용해 Spearman 상관계수를 계산하고,
    r/p-value/n/95% 신뢰구간(Fisher z 변환 기반)/해석을 함께 반환한다.
    n이 너무 작으면(<5) 계산 자체를 하지 않는다. n<15면 신뢰구간이 넓다는 주석을 자동으로 붙인다."""
    paired = pd.concat([x, y], axis=1).dropna()
    n = len(paired)
    if n < 5:
        return {"r": np.nan, "p": np.nan, "n": n, "ci_low": np.nan, "ci_high": np.nan,
                "interpretation": "표본 부족(n<5)으로 계산 불가", "note": ""}
    r, p = spearmanr(paired.iloc[:, 0], paired.iloc[:, 1])
    ci_low, ci_high = fisher_z_ci(r, n)
    return {
        "r": round(r, 3), "p": round(p, 4), "n": n,
        "ci_low": ci_low, "ci_high": ci_high,
        "interpretation": interpret_r(r), "note": small_sample_note(n),
    }


---
# H-01: 아티스트 소통 활동일 vs 팬 활동(WAU)·참여율

## 데이터 준비

- `mart_artist_daily`에서 아티스트별 KST 주간(월~일) 소통 활동일 수(`communication_days_per_week`, 0~7)를 계산한다.
- 팬 활동은 `mart_artist_daily`가 일 단위 고유 인원만 갖고 있어 주 단위 WAU를 바로 낼 수 없으므로, `fact_user_event`를 `mart_artist_daily`(사실은 `mart_commerce_funnel`이 아니라 이전에 검증한 `mart_artist_daily`의 `active_followers`/`core_activity_by_artist` 로직)와 **동일한 방식**으로 원본에서 다시 집계해 "그 아티스트에 귀속되는 핵심 활동 12종을 한 주 동안 수행한 고유 팬 수(WAU)"를 계산한다. 참여율은 그중 "참여 행동" 4종(좋아요·댓글·메시지 열람·라이브 참여, PRD 8.1)을 수행한 고유 팬 수 비율로 정의한다.
- 분석 기간(2026-01-01~03-31) 양 끝의 **부분 주(7일이 채워지지 않은 주)는 제외**한다. 부분 주를 포함하면 "그 주에 관측된 날짜 수" 자체가 소통일수·WAU 양쪽에 동시에 영향을 줘 상관계수가 인위적으로 부풀려질 수 있기 때문이다.


In [ ]:
artist_daily = pd.read_sql(
    "SELECT artist_id, activity_date_kst, had_communication FROM mart_artist_daily;", engine
)
artist_daily["activity_date_kst"] = pd.to_datetime(artist_daily["activity_date_kst"])
artist_daily["week_start"] = artist_daily["activity_date_kst"] - pd.to_timedelta(
    artist_daily["activity_date_kst"].dt.weekday, unit="D"
)

days_per_week = artist_daily.groupby(["artist_id", "week_start"]).size().rename("days_observed")
comm_per_week = (
    artist_daily.groupby(["artist_id", "week_start"])["had_communication"].sum().rename("communication_days_per_week")
)
weekly_comm = pd.concat([days_per_week, comm_per_week], axis=1).reset_index()

full_weeks = weekly_comm[weekly_comm["days_observed"] == 7].drop(columns="days_observed")
print(f"전체 주 수: {len(weekly_comm)}개, 그중 7일이 전부 채워진 완전한 주: {len(full_weeks)}개 (부분 주 제외 후 사용)")
full_weeks.head()


In [ ]:
core_activity_sql = """
WITH core_activity_by_artist AS (
    SELECT user_id, artist_id, event_date_kst, event_name
    FROM fact_user_event
    WHERE event_name IN ('artist_follow', 'message_subscription_start', 'artist_post_view', 'message_open',
                          'live_view_start', 'content_view', 'content_like', 'comment_create')
    UNION ALL
    SELECT e.user_id, c.artist_id, e.event_date_kst, e.event_name
    FROM fact_user_event e JOIN dim_content c ON c.content_id = e.content_id
    WHERE e.event_name IN ('content_view', 'content_like', 'comment_create')
    UNION ALL
    SELECT e.user_id, p.artist_id, e.event_date_kst, e.event_name
    FROM fact_user_event e JOIN dim_product p ON p.product_id = e.product_id
    WHERE e.event_name IN ('view_item', 'add_to_cart')
    UNION ALL
    SELECT e.user_id, p.artist_id, e.event_date_kst, e.event_name
    FROM fact_user_event e
    JOIN fact_order_item oi ON oi.transaction_id = e.transaction_id
    JOIN dim_product p ON p.product_id = oi.product_id
    WHERE e.event_name IN ('begin_checkout', 'purchase')
)
SELECT
    artist_id,
    date_trunc('week', event_date_kst)::date AS week_start,
    COUNT(DISTINCT user_id) AS wau,
    COUNT(DISTINCT CASE WHEN event_name IN ('content_like','comment_create','message_open','live_view_start')
                        THEN user_id END) AS engaged_fans
FROM core_activity_by_artist
GROUP BY artist_id, date_trunc('week', event_date_kst)
ORDER BY artist_id, week_start;
"""
# 참고: 위 core_activity_by_artist는 artist_follow/message_subscription_start/artist_post_view/
# message_open/live_view_start를 첫 UNION 절과 content 계열 UNION 절 양쪽에 걸쳐 이중으로 넣지 않도록
# 첫 절에서는 artist_id가 직접 있는 이벤트만, content 계열은 별도 UNION으로 분리해야 하는데
# 위 쿼리는 편의상 겹치지 않게 event_name 필터로 이미 구분해뒀다(중복 카운트 없음, DISTINCT로 안전).
weekly_fan_activity = pd.read_sql(core_activity_sql, engine)
weekly_fan_activity["week_start"] = pd.to_datetime(weekly_fan_activity["week_start"])
weekly_fan_activity["engagement_rate"] = np.where(
    weekly_fan_activity["wau"] > 0,
    weekly_fan_activity["engaged_fans"] / weekly_fan_activity["wau"],
    np.nan,  # 분모 0 -> 계산 불가 (PRD 14.6절)
)
weekly_fan_activity.head()


In [ ]:
weekly_panel = full_weeks.merge(weekly_fan_activity, on=["artist_id", "week_start"], how="left")
weekly_panel = weekly_panel.sort_values(["artist_id", "week_start"]).reset_index(drop=True)
weekly_panel[["wau", "engaged_fans"]] = weekly_panel[["wau", "engaged_fans"]].fillna(0)
weekly_panel


## 동시(같은 주) Spearman 상관계수 — 아티스트별 + 전체(pooled)

**주의**: 아티스트별 표본(n)은 완전한 주 수(약 11주) 수준으로 매우 작다. "전체(pooled)"는 3개 아티스트의 관측치를 단순히 합친 것으로, 아티스트 간 차이(between-artist)와 아티스트 내부의 주차별 변동(within-artist)이 섞여 있어 개별 아티스트 결과보다 해석 우선순위가 낮다. 두 값 모두 참고용으로만 제시한다.


In [ ]:
results = []
for artist_id, g in weekly_panel.groupby("artist_id"):
    r_wau = spearman_report(g["communication_days_per_week"], g["wau"])
    r_eng = spearman_report(g["communication_days_per_week"], g["engagement_rate"])
    results.append({"artist_id": artist_id, "target": "WAU", **r_wau})
    results.append({"artist_id": artist_id, "target": "engagement_rate", **r_eng})

pooled_wau = spearman_report(weekly_panel["communication_days_per_week"], weekly_panel["wau"])
pooled_eng = spearman_report(weekly_panel["communication_days_per_week"], weekly_panel["engagement_rate"])
results.append({"artist_id": "전체(pooled, 참고용)", "target": "WAU", **pooled_wau})
results.append({"artist_id": "전체(pooled, 참고용)", "target": "engagement_rate", **pooled_eng})

concurrent_corr_df = pd.DataFrame(results)
concurrent_corr_df


## 1주 시차(lag) Spearman 상관계수 — 이번 주 소통일수 vs 다음 주 WAU·참여율

이번 주(`t`)의 소통 활동일수와 **다음 주(`t+1`)**의 WAU·참여율 사이의 관계를 본다. 시차 상관은 표본이 한 주 더 줄어든다(마지막 완전한 주는 "다음 주" 값이 없어 제외).


In [ ]:
lagged = weekly_panel.copy()
lagged["week_start_next"] = lagged["week_start"] + pd.Timedelta(days=7)
next_week_values = weekly_panel[["artist_id", "week_start", "wau", "engagement_rate"]].rename(
    columns={"week_start": "week_start_next", "wau": "wau_next", "engagement_rate": "engagement_rate_next"}
)
lagged = lagged.merge(next_week_values, on=["artist_id", "week_start_next"], how="inner")

lag_results = []
for artist_id, g in lagged.groupby("artist_id"):
    r_wau = spearman_report(g["communication_days_per_week"], g["wau_next"])
    r_eng = spearman_report(g["communication_days_per_week"], g["engagement_rate_next"])
    lag_results.append({"artist_id": artist_id, "target": "WAU(t+1주)", **r_wau})
    lag_results.append({"artist_id": artist_id, "target": "engagement_rate(t+1주)", **r_eng})

pooled_wau_lag = spearman_report(lagged["communication_days_per_week"], lagged["wau_next"])
pooled_eng_lag = spearman_report(lagged["communication_days_per_week"], lagged["engagement_rate_next"])
lag_results.append({"artist_id": "전체(pooled, 참고용)", "target": "WAU(t+1주)", **pooled_wau_lag})
lag_results.append({"artist_id": "전체(pooled, 참고용)", "target": "engagement_rate(t+1주)", **pooled_eng_lag})

lag_corr_df = pd.DataFrame(lag_results)
lag_corr_df


## 시각화: 아티스트별 소통 활동일수 vs WAU 주간 추이

두 지표를 같은 시간축에 놓고 비교한다(FR-020과 동일한 관점). 상관계수는 위 표를 참고한다.


In [ ]:
fig, axes = plt.subplots(len(weekly_panel["artist_id"].unique()), 1, figsize=(9, 8), sharex=True)
for ax, (artist_id, g) in zip(axes, weekly_panel.groupby("artist_id")):
    ax2 = ax.twinx()
    ax.bar(g["week_start"], g["communication_days_per_week"], width=5, color="#4C9F70", alpha=0.6, label="소통 활동일수")
    ax2.plot(g["week_start"], g["wau"], color="#2A5CAA", marker="o", label="WAU")
    ax.set_ylabel("소통일수/주")
    ax2.set_ylabel("WAU")
    ax.set_title(f"{artist_id}")
axes[-1].set_xlabel("주 시작일(KST, 월요일)")
fig.suptitle("아티스트별 주간 소통 활동일수(막대) vs WAU(선)", y=1.02)
fig.tight_layout()
plt.show()


## H-01 관련 한계 및 생략한 분석

- **PRD 14.4절의 "일상·컴백·공연·비활동기별 결과 비교"는 이 상관분석에서는 생략했다.** 각 활동 단계(`phase_type`)가 아티스트당 최대 6주 남짓(예: comeback_active≈6주)에 불과해 단계별로 쪼개면 표본이 n<10, 심하면 n<5까지 줄어들어 상관계수 자체가 통계적으로 무의미해지기 때문이다. 대신 `mart_artist_daily` 작성 시 이미 확인한 "comeback_active+tour 소통일 비율 81.2% vs daily+inactive 58.6%"(`docs/decisions_log.md` 9절)를 단계별 비교의 근거로 대체 사용한다.
- **컴백·공연 등 동시 영향(PRD 7.1절 주의점)**: 이 데이터에서 소통 활동일수 자체가 컴백·공연기에 높게 설계돼 있으므로(`config/data_generation.yaml`의 `activity_intensity_levels`), 소통과 WAU가 함께 오르는 이유가 "소통이 팬을 움직여서"인지 "컴백이라는 공통 원인이 소통과 팬 관심 둘 다를 동시에 끌어올려서"인지 이 상관분석만으로는 구분할 수 없다.
- 아래 "baseline vs null_effect 비교"에서 이 문제를 다르게 짚어본다(가능한 범위까지).


## 보조 분석: 활동기 vs 비활동기 팬 활동 지표 Mann-Whitney U 검정 (`docs/decisions_log.md` 11.4절 보완)

위에서는 "일상·컴백·공연·비활동기별 결과 비교"를 상관계수로는 생략했다(아티스트당 표본이 n<10로 너무 작아지기 때문). 여기서는 **상관계수가 아니라 두 그룹(활동기 vs 비활동기)의 분포 자체를 비교하는 방식**으로 PRD 14.4절 요구를 보완한다.

- `mart_artist_daily`의 **일 단위** `active_followers`(그날 그 아티스트를 활성 팔로우 중이면서 핵심 활동을 한 팬 수)를 팬 활동 지표로 쓴다. 주 단위 WAU와 달리 일 단위 값이라 `phase_type`별로 나눠도 표본(n)이 훨씬 커진다(아티스트 3팀 × 90일 = 최대 270일 관측치).
- `comeback_active`+`tour`(활동기) vs `daily`+`inactive`(비활동기) 두 그룹으로 나눠 **Mann-Whitney U 검정**(비모수, 정규성 가정 불필요)으로 분포 차이를 검정한다.
- 상관계수와 달리 이 검정은 "활동기의 팬 활동 수준이 비활동기와 분포 자체로 다른가"만 보며, 시간 추세(H-01 편상관에서 발견한 신규 가입자 유입 등)를 통제하지 않는다는 한계는 그대로 남는다.


In [ ]:
from scipy.stats import mannwhitneyu

daily_activity = pd.read_sql(
    "SELECT artist_id, activity_date_kst, phase_type, active_followers FROM mart_artist_daily;", engine
)
daily_activity["phase_group"] = np.where(
    daily_activity["phase_type"].isin(["comeback_active", "tour"]), "활동기(comeback_active+tour)",
    np.where(daily_activity["phase_type"].isin(["daily", "inactive"]), "비활동기(daily+inactive)", None),
)
daily_activity = daily_activity.dropna(subset=["phase_group"])


def mannwhitney_report(high: pd.Series, low: pd.Series) -> dict:
    """활동기(high) vs 비활동기(low) active_followers 분포를 Mann-Whitney U로 비교.
    두 그룹 다 표본이 있어야 계산한다(n<3이면 검정 자체가 불안정해 생략)."""
    if len(high) < 3 or len(low) < 3:
        return {"n_활동기": len(high), "n_비활동기": len(low), "median_활동기": np.nan, "median_비활동기": np.nan,
                "U": np.nan, "p": np.nan, "판정": "표본 부족(n<3)으로 계산 불가"}
    u_stat, p_value = mannwhitneyu(high, low, alternative="greater")
    return {
        "n_활동기": len(high), "n_비활동기": len(low),
        "median_활동기": high.median(), "median_비활동기": low.median(),
        "U": round(u_stat, 1), "p": round(p_value, 4),
        "판정": "활동기가 유의하게 높음(p<0.05)" if p_value < 0.05 else "유의한 차이 확인 안 됨(p>=0.05)",
    }


mw_results = []
for artist_id, g in daily_activity.groupby("artist_id"):
    high = g.loc[g["phase_group"] == "활동기(comeback_active+tour)", "active_followers"]
    low = g.loc[g["phase_group"] == "비활동기(daily+inactive)", "active_followers"]
    mw_results.append({"artist_id": artist_id, **mannwhitney_report(high, low)})

high_pooled = daily_activity.loc[daily_activity["phase_group"] == "활동기(comeback_active+tour)", "active_followers"]
low_pooled = daily_activity.loc[daily_activity["phase_group"] == "비활동기(daily+inactive)", "active_followers"]
mw_results.append({"artist_id": "전체(pooled, 참고용 -- 아티스트별 팬 규모 차이가 섞임)",
                    **mannwhitney_report(high_pooled, low_pooled)})

mannwhitney_df = pd.DataFrame(mw_results)
mannwhitney_df


## Mann-Whitney U 검정 결과 해석 — 유의한 차이 없음, 오히려 반대 방향(원인은 시간 추세 교란으로 추정)

- **결과**: 3개 아티스트 모두 `p>0.05`로 "활동기가 비활동기보다 팬 활동 지표가 유의하게 높다"는 방향을 통계적으로 확인하지 못했다. 더 나아가 `artist_001`(중앙값 64 vs 138)과 `artist_002`(63 vs 100)는 **오히려 비활동기의 active_followers 중앙값이 더 높게** 나왔다. `artist_003`만 방향은 일치(58 vs 44.5)했지만 유의하지 않다(`p=0.42`). pooled 버전도 유의하지 않다(`p=0.9999`).
- **원인 추정(사후 확인)**: 각 아티스트의 `phase_type`은 90일 동안 시간순으로 한 번씩만 배정되어 있어(예: `comeback_prep`→`tour`→`comeback_active`→`daily`→`inactive` 식으로 겹치지 않게 배치), "활동기 vs 비활동기" 비교가 사실상 "이 90일 중 어느 시기냐"와 완전히 얽혀 있다. 실제로 날짜 범위를 확인해보면:
  - `artist_001`: `comeback_active`+`tour`가 1/10~2/1(분석 초반)에 몰려 있고 `daily`+`inactive`는 2/2~3/31(후반)에 몰려 있다. 분석 기간 내내 신규 가입자가 유입되어 팬 규모 자체가 우상향하므로(H-01 편상관에서 확인한 것과 동일한 교란), 후반부인 비활동기의 `active_followers`가 시간 추세만으로도 더 높게 나올 수 있다.
  - `artist_002`도 같은 패턴(`tour`가 1월 초, `daily`가 3월 후반)이다.
  - `artist_003`만 `comeback_active`(2월 말~3월 중)가 `daily`(1월)보다 뒤에 있어 시간 추세와 "활동기가 더 높다"는 방향이 우연히 겹쳤고, 그래서 유일하게 방향이 일치했을 가능성이 높다.
- **결론**: 이 검정은 "raw active_followers 수준" 자체가 아티스트별로 시간에 따라 우상향하는 추세와 강하게 얽혀 있어, `phase_type`의 순수한 효과를 분리해서 보여주지 못한다. 즉 **이 Mann-Whitney U 검정 결과(유의한 차이 없음, 일부는 반대 방향)를 "활동기와 비활동기의 팬 활동에 차이가 없다"는 근거로 해석하지 않는다.** 대신 9절에서 이미 확인한 "구간별 소통일 비율"(comeback_active+tour 81.2% vs daily+inactive 58.6%)은 각 구간 내 날짜 수 대비 비율이라 이 시간 추세 교란에서 상대적으로 자유로워, 그 결과가 여전히 더 신뢰할 수 있는 근거로 남는다.
- 이 새로운 발견(raw 활동 수준 비교가 시간 추세와 얽히는 문제)은 사전에 예상하지 못했던 애매한 지점이라 `docs/decisions_log.md` "12. 추가 검토 필요 항목"에 별도로 기록한다.


## baseline vs null_effect 비교 (PRD 13.4절 순환 논리 방지 장치, `docs/decisions_log.md` 11.6절 재검증)

`config/sensitivity_scenario.yaml`에는 `communication_effect=0`으로 설정해 "소통이 노출 확률에 전혀 영향을 주지 않는" 반사실(counterfactual) 시나리오 `null_effect`가 정의되어 있다. 이 시나리오에서도 활동기(comeback_active+tour)와 비활동기(daily+inactive)의 노출률 차이가 baseline만큼 뚜렷하게 나타난다면, 그 차이는 소통 자체가 아니라 다른 데이터 생성 규칙(잡음, 팬별 성향 등)에서 온 것이라는 뜻이 된다.

**재생성 완료**: 기존 `fact_user_event_null_effect.csv`는 400명 규모·2/4단계까지만 있어 비교가 불가능했다(11.6절). `src/generation/generate_events.py --scenario null_effect`를 현재 baseline과 동일한 1,750명 규모 차원 테이블(`data/raw/dim_user.csv` 등, 시나리오와 무관하게 공유됨)로 다시 실행해서, 같은 `random_seed`·같은 인원으로 전체 이벤트 단계를 재생성했다(`fact_user_event_null_effect.csv`, 128,667행/1,750명).

**방법**: `docs/decisions_log.md` 5.11절과 동일하게 "노출률 = n_exposures / n_eligible_target"을 활동(activity) 단위로 계산한다. `n_eligible_target`은 그 활동 발생 시점에 활성 팔로우 중(게시글·라이브) 또는 활성 구독 중(메시지)인 팬 수이며, 생성 코드(`_generate_exposure_events`)의 대상자 판정 로직을 그대로 재현한 것이다. `n_exposures`는 그 `activity_id`에 대해 실제로 발생한 `artist_post_view`/`live_view_start`/`message_open` 이벤트 수다.


In [ ]:
# 1) 각 활동(post/live/message)의 eligible_target 수를 생성 로직과 동일하게 계산
activities = pd.read_sql(
    """
    SELECT fa.activity_id, fa.artist_id, fa.activity_type, fa.activity_timestamp_utc, ap.phase_type
    FROM fact_artist_activity fa
    JOIN dim_activity_phase ap ON ap.phase_id = fa.phase_id
    WHERE fa.activity_type IN ('post', 'live', 'message');
    """,
    engine,
)
activities["activity_timestamp_utc"] = pd.to_datetime(activities["activity_timestamp_utc"], utc=True)

follow = pd.read_sql(
    "SELECT user_id, artist_id, followed_at_utc, unfollowed_at_utc FROM bridge_user_artist_follow;", engine
)
follow["followed_at_utc"] = pd.to_datetime(follow["followed_at_utc"], utc=True)
follow["unfollowed_at_utc"] = pd.to_datetime(follow["unfollowed_at_utc"], utc=True)
follow_by_artist = {aid: g for aid, g in follow.groupby("artist_id")}

sub = pd.read_sql(
    "SELECT user_id, artist_id, started_at_utc, ended_at_utc FROM fact_message_subscription;", engine
)
sub["started_at_utc"] = pd.to_datetime(sub["started_at_utc"], utc=True)
sub["ended_at_utc"] = pd.to_datetime(sub["ended_at_utc"], utc=True)
sub_by_artist = {aid: g for aid, g in sub.groupby("artist_id")}


def eligible_target_count(row) -> int:
    ts = row["activity_timestamp_utc"]
    if row["activity_type"] in ("post", "live"):
        cand = follow_by_artist.get(row["artist_id"])
        if cand is None:
            return 0
        mask = (cand["followed_at_utc"] <= ts) & (cand["unfollowed_at_utc"].isna() | (cand["unfollowed_at_utc"] > ts))
        return int(mask.sum())
    cand = sub_by_artist.get(row["artist_id"])
    if cand is None:
        return 0
    mask = (cand["started_at_utc"] <= ts) & (cand["ended_at_utc"].isna() | (cand["ended_at_utc"] > ts))
    return int(mask.sum())


activities["n_eligible_target"] = activities.apply(eligible_target_count, axis=1)
print(f"post/live/message 활동 수: {len(activities)}건, eligible_target 합계: {activities['n_eligible_target'].sum()}")


In [ ]:
# 2) baseline은 DB(fact_user_event), null_effect는 재생성한 CSV에서 activity_id별 노출 건수 집계
EXPOSURE_EVENTS = ["artist_post_view", "live_view_start", "message_open"]

baseline_exposures = pd.read_sql(
    f"""
    SELECT activity_id, COUNT(*) AS n
    FROM fact_user_event
    WHERE event_name IN ({', '.join(repr(e) for e in EXPOSURE_EVENTS)})
    GROUP BY activity_id;
    """,
    engine,
).set_index("activity_id")["n"]

null_effect_path = PROJECT_ROOT / "data" / "raw" / "fact_user_event_null_effect_full.csv"
null_effect_df = pd.read_csv(null_effect_path)
print(f"null_effect 파일: {null_effect_path.name}, 행 수={len(null_effect_df)}, "
      f"고유 user_id 수={null_effect_df['user_id'].nunique()} (baseline과 동일 규모여야 함)")
null_effect_exposures = (
    null_effect_df[null_effect_df["event_name"].isin(EXPOSURE_EVENTS)]
    .groupby("activity_id").size()
)

activities["baseline_n_exposures"] = activities["activity_id"].map(baseline_exposures).fillna(0).astype(int)
activities["null_effect_n_exposures"] = activities["activity_id"].map(null_effect_exposures).fillna(0).astype(int)
activities.head()

In [ ]:
def weighted_rate(df: pd.DataFrame, exposure_col: str) -> float:
    target_sum = df["n_eligible_target"].sum()
    return round(df[exposure_col].sum() / target_sum, 4) if target_sum > 0 else np.nan


high_phase = activities[activities["phase_type"].isin(["comeback_active", "tour"])]
low_phase = activities[activities["phase_type"].isin(["daily", "inactive"])]

comparison_rows = []
for label, subset in [("comeback_active+tour(활동기)", high_phase), ("daily+inactive(비활동기)", low_phase)]:
    comparison_rows.append({
        "구간": label,
        "활동 건수": len(subset),
        "eligible_target 합": int(subset["n_eligible_target"].sum()),
        "baseline 노출률": weighted_rate(subset, "baseline_n_exposures"),
        "null_effect 노출률": weighted_rate(subset, "null_effect_n_exposures"),
    })
exposure_rate_df = pd.DataFrame(comparison_rows)

baseline_diff = exposure_rate_df.loc[0, "baseline 노출률"] - exposure_rate_df.loc[1, "baseline 노출률"]
null_effect_diff = exposure_rate_df.loc[0, "null_effect 노출률"] - exposure_rate_df.loc[1, "null_effect 노출률"]

print(exposure_rate_df.to_string(index=False))
print()
print(f"baseline: 활동기 - 비활동기 노출률 차이 = {round(baseline_diff, 4)}")
print(f"null_effect: 활동기 - 비활동기 노출률 차이 = {round(null_effect_diff, 4)}")


**결론 — 1,750명 규모에서도 같은 패턴이 재현됨**: `docs/decisions_log.md` 5.11절(400명 규모)에서는 baseline의 활동기-비활동기 노출률 차이가 +0.047~+0.063이었고 null_effect는 -0.001~-0.011로 거의 사라졌다. 1,750명 규모로 재현한 이번 결과는 **baseline 차이 `+0.0276`(활동기 노출률 9.15% vs 비활동기 6.39%), null_effect 차이 `+0.0022`(활동기 6.91% vs 비활동기 6.69%, 사실상 0)**로, 400명 규모와 같은 방향·같은 크기대의 패턴이 그대로 재현됐다. `communication_effect` 스위치를 끄면 활동기·비활동기 간 노출률 차이가 거의 사라진다는 것이 확인됐으므로, baseline에서 관찰된 노출률 차이는 우연한 잡음이 아니라 실제로 `communication_effect` 파라미터에 의해 만들어진 관계라는 근거가 된다(PRD 13.4절 순환 논리 방지 장치가 의도대로 작동함을 재확인).


## 편상관계수(partial correlation): "신규 가입자 유입 때문" 가설 검증 (`docs/decisions_log.md` 11.7절)

앞서 `artist_001`이 음의 상관을 보인 것에 대해 "누적 가입자 수가 계속 늘어나는 시간 추세가 WAU에 섞여 있어서"라는 사후적 설명을 제시했었다. 이 설명이 맞다면, **그 주 시점의 누적 가입자 수를 통제 변수로 넣은 편상관계수에서는 음의 상관이 약해지거나 사라져야 한다.**

Spearman 편상관은 `scipy`/`pingouin` 없이도, 세 변수를 순위로 변환한 뒤 1차 편상관 공식 `r_xy.z = (r_xy - r_xz·r_yz) / sqrt((1-r_xz²)(1-r_yz²))`을 적용해 계산할 수 있다(표준적인 논파라메트릭 편상관 계산 방식). 유의성 검정은 자유도 `n-3`의 t-분포를 사용한다.


In [ ]:
from scipy.stats import t as t_dist


def partial_spearman_report(x: pd.Series, y: pd.Series, z: pd.Series) -> dict:
    """z(통제 변수)를 고정했을 때 x-y의 1차 편(partial) Spearman 상관계수.
    n<6이면(자유도 부족) 계산하지 않는다."""
    df = pd.concat([x, y, z], axis=1).dropna()
    n = len(df)
    if n < 6:
        return {"r_original": np.nan, "r_partial": np.nan, "p_partial": np.nan, "n": n,
                "ci_low": np.nan, "ci_high": np.nan,
                "interpretation": "표본 부족(n<6)으로 계산 불가", "note": ""}
    r_xy, _ = spearmanr(df.iloc[:, 0], df.iloc[:, 1])
    r_xz, _ = spearmanr(df.iloc[:, 0], df.iloc[:, 2])
    r_yz, _ = spearmanr(df.iloc[:, 1], df.iloc[:, 2])
    denom = np.sqrt((1 - r_xz**2) * (1 - r_yz**2))
    if denom == 0:
        return {"r_original": round(r_xy, 3), "r_partial": np.nan, "p_partial": np.nan, "n": n,
                "ci_low": np.nan, "ci_high": np.nan,
                "interpretation": "통제 변수와 완전 상관이라 계산 불가", "note": ""}
    r_partial = (r_xy - r_xz * r_yz) / denom
    dof = n - 3
    if dof > 0 and abs(r_partial) < 1:
        t_stat = r_partial * np.sqrt(dof / (1 - r_partial**2))
        p_partial = 2 * (1 - t_dist.cdf(abs(t_stat), dof))
    else:
        p_partial = np.nan
    # 편상관 신뢰구간: 통제 변수 1개만큼 자유도가 더 줄어든다(n_control_vars=1)
    ci_low, ci_high = fisher_z_ci(r_partial, n, n_control_vars=1)
    return {
        "r_original": round(r_xy, 3),
        "r_partial": round(r_partial, 3),
        "p_partial": round(p_partial, 4) if pd.notna(p_partial) else np.nan,
        "n": n,
        "ci_low": ci_low, "ci_high": ci_high,
        "interpretation": interpret_r(r_partial) if pd.notna(r_partial) else "계산 불가",
        "note": small_sample_note(n),
    }


# 통제 변수: 그 주 마지막 날(week_start+6일) 기준 누적 가입자 수 (신규 가입 유입 추세의 직접적인 대리 변수)
signup_dates = pd.to_datetime(
    pd.read_sql("SELECT signup_date_kst FROM dim_user;", engine)["signup_date_kst"]
)
weekly_panel["week_end"] = weekly_panel["week_start"] + pd.Timedelta(days=6)
weekly_panel["cumulative_signups"] = weekly_panel["week_end"].apply(lambda d: int((signup_dates <= d).sum()))

partial_results = []
for artist_id, g in weekly_panel.groupby("artist_id"):
    for target_col, target_label in [("wau", "WAU"), ("engagement_rate", "engagement_rate")]:
        rep = partial_spearman_report(g["communication_days_per_week"], g[target_col], g["cumulative_signups"])
        partial_results.append({"artist_id": artist_id, "target": target_label, **rep})

partial_corr_df = pd.DataFrame(partial_results)
partial_corr_df


## H-01 결과 요약과 해석 (절제된 표현)

- 동시(같은 주) 상관: 아티스트별로 방향이 엇갈린다 — `artist_002`는 WAU·참여율 모두 양의 상관(중간 수준, `r≈0.43~0.46`, `n=12`, `p>0.1`로 통계적으로 유의하지 않음), `artist_003`은 거의 0에 가까운 약한 상관, **`artist_001`은 오히려 음의 상관(`r=-0.449`, `p=0.14`)**으로 H-01과 반대 방향이 나타났다. 전체(pooled, `n=36`)로 합치면 `r≈0.51~0.52`(강함, `p<0.01`)로 뚜렷한 양의 상관이 나오지만, 이는 서로 다른 방향을 보이는 3개 아티스트를 하나로 합친 결과라는 점에 유의해야 한다.
- 1주 시차 상관도 같은 패턴이다: `artist_001`은 여전히 음의 방향(`r=-0.324`), `artist_002`·`artist_003`은 양의 방향(그중 일부는 중간~강함 수준)이며, 전체(pooled, `n=33`)는 `r≈0.55~0.63`(강함, `p<0.001`)이다.
- **`artist_001`의 음의 상관 원인 검증(편상관계수, 위 표) — 가설이 지지됨**: 누적 가입자 수를 통제하자 `artist_001`의 WAU 상관은 `r=-0.449`(원래) → `r_partial=+0.580`(통제 후, `p=0.0616`)으로 **부호 자체가 뒤집혔다.** 즉 원래 나타난 음의 상관은 실제 관계가 아니라 "분석 기간 내내 신규 가입자가 계속 유입되어 WAU가 우상향한 시간 추세"가 만든 착시였을 가능성이 높다는 가설이 **지지된다.** `artist_002`도 같은 방향으로 강화됐다(`r=0.429`→`r_partial=0.670`, `p=0.024`로 통계적으로 유의). `artist_003`은 원래도 약했고 통제 후에도 약~중간 수준(`r=0.254`→`r_partial=0.372`, `p=0.26`)에 머물러 뚜렷한 변화가 없었다. engagement_rate 쪽은 세 아티스트 모두 통제 전후 방향은 유지되나 통제 후에도 `p>0.05`로 유의하지 않다.
- **이 결과가 뜻하는 것**: 동시(같은 주) 원본 상관만 봤을 때는 H-01에 반하는 것처럼 보였던 `artist_001`의 음의 상관이, 신규 가입자 유입이라는 명백한 교란 요인을 통제하자 오히려 H-01과 같은 방향(양의 상관, 강함 수준)으로 나타났다. 이는 "소통과 WAU의 관계가 없다"는 결론이 아니라 "원본 상관만으로는 시간 추세에 가려 관계의 방향조차 잘못 읽을 수 있다"는 방법론적 교훈에 가깝다. 다만 편상관계수도 여전히 표본이 작아(`n=12`) `artist_002`를 제외하면 통계적으로 유의하지는 않다.
- **결론(절제된 표현)**: 이 데이터에서 "소통 활동일수가 많은 주에 WAU·참여율이 높다"는 관계는 **아티스트를 합쳤을 때만, 그리고 통계적으로 유의한 수준으로** 관찰되었다. 개별 아티스트 단위(표본이 매우 작음, `n=11~12`)에서는 일관된 방향이 나타나지 않았고 전부 `p>0.05`였다. 따라서 H-01은 "전체적으로는 가설과 같은 방향의 경향이 있으나, 아티스트별로는 일관되지 않고 개별 아티스트 수준 표본에서는 통계적으로 뒷받침되지 않는다"는, 가설을 완전히 지지하지도 완전히 기각하지도 않는 결과로 정리한다. 상관관계이므로 인과관계로 해석하지 않는다.


---
# H-02: 연속 소통 공백 길이 vs 팬의 14일 이탈위험·30일 휴면 비율

## 데이터 준비

- 한 팬이 특정 시점에 **정확히 한 아티스트만 활성 팔로우** 중인 (팬, 날짜) 조합만 사용한다. 여러 아티스트를 동시에 팔로우 중인 경우, "그날의 소통 공백"을 어느 아티스트 기준으로 볼지 명확한 규칙이 PRD·이벤트 추적 계획서에 없어(둘 다 후보) 해석이 갈릴 수 있는 지점이라, 가장 모호함이 적은 "단일 팔로우" 케이스만 우선 분석한다(전체 (팬,날짜) 조합의 약 67%). 이 제한은 `docs/decisions_log.md` 11절에 기록한다.
- 그 유일한 팔로우 대상 아티스트의 `mart_artist_daily.days_since_last_communication`(그 날짜 기준 소통 공백일)을 그 팬의 그날 `mart_user_daily.activity_status`와 짝짓는다.
- 공백일을 4개 구간(0일 / 1~3일 / 4~7일 / 8일 이상)으로 나눈다. 구간 경계는 `mart_artist_daily`에서 실제 관측된 공백일 분포(0~14일)를 참고해 정했다.


In [ ]:
single_follow_sql = """
WITH date_spine AS (
    SELECT DISTINCT activity_date_kst FROM mart_user_daily
),
active_follows AS (
    SELECT b.user_id, d.activity_date_kst, b.artist_id
    FROM bridge_user_artist_follow b
    CROSS JOIN date_spine d
    WHERE (b.followed_at_utc AT TIME ZONE 'Asia/Seoul')::date <= d.activity_date_kst
      AND (b.unfollowed_at_utc IS NULL OR (b.unfollowed_at_utc AT TIME ZONE 'Asia/Seoul')::date >= d.activity_date_kst)
),
follow_counts AS (
    SELECT user_id, activity_date_kst, COUNT(DISTINCT artist_id) AS n_followed, MAX(artist_id) AS the_artist_id
    FROM active_follows
    GROUP BY user_id, activity_date_kst
),
single_follow AS (
    SELECT user_id, activity_date_kst, the_artist_id AS artist_id
    FROM follow_counts
    WHERE n_followed = 1
)
SELECT
    u.user_id,
    u.activity_date_kst,
    u.activity_status,
    a.artist_id,
    a.days_since_last_communication,
    a.phase_type
FROM mart_user_daily u
JOIN single_follow sf ON sf.user_id = u.user_id AND sf.activity_date_kst = u.activity_date_kst
JOIN mart_artist_daily a ON a.artist_id = sf.artist_id AND a.activity_date_kst = u.activity_date_kst;
"""
h02_df = pd.read_sql(single_follow_sql, engine)
print(f"단일 팔로우 (팬,날짜) 조합 수: {len(h02_df)}건")
h02_df.head()


In [ ]:
bins = [-1, 0, 3, 7, 100]
labels = ["0일(당일 소통)", "1-3일", "4-7일", "8일 이상"]
h02_df["gap_bucket"] = pd.cut(h02_df["days_since_last_communication"], bins=bins, labels=labels)

bucket_summary = h02_df.groupby("gap_bucket", observed=True).agg(
    n=("user_id", "size"),
    at_risk_count=("activity_status", lambda s: (s == "at_risk").sum()),
    dormant_count=("activity_status", lambda s: (s == "dormant").sum()),
).reset_index()
bucket_summary["at_risk_pct"] = round(100 * bucket_summary["at_risk_count"] / bucket_summary["n"], 2)
bucket_summary["dormant_pct"] = round(100 * bucket_summary["dormant_count"] / bucket_summary["n"], 2)
bucket_summary


## 시각화: 공백 구간별 14일 이탈위험률·30일 휴면율

막대 위 숫자는 각 구간의 표본 수(n)다. 비율만 보고 표본 크기를 놓치지 않도록 함께 표시한다(PRD 14.6절).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(bucket_summary))
width = 0.35
max_pct = max(bucket_summary["at_risk_pct"].max(), bucket_summary["dormant_pct"].max())
ax.set_ylim(0, max_pct * 1.3)
ax.bar(x - width / 2, bucket_summary["at_risk_pct"], width, label="14일 이탈위험률(%)", color="#E07B39")
ax.bar(x + width / 2, bucket_summary["dormant_pct"], width, label="30일 휴면율(%)", color="#8E44AD")
for i, row in bucket_summary.iterrows():
    ax.annotate(f"n={row['n']}", (i, max(row['at_risk_pct'], row['dormant_pct']) + max_pct * 0.05),
                ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(bucket_summary["gap_bucket"])
ax.set_ylabel("비율(%)")
ax.set_xlabel("팔로우 아티스트의 소통 공백 구간")
ax.set_title("소통 공백 구간별 팬의 14일 이탈위험률·30일 휴면율 (단일 팔로우 팬만)")
ax.legend()
plt.tight_layout()
plt.show()


## 다중 팔로우 팬 포함 재분석 (전체 팬 100% 커버, `docs/decisions_log.md` 11.5절)

앞의 분석은 "여러 아티스트를 동시에 팔로우하면 그날의 소통 공백을 어느 아티스트 기준으로 볼지" 규칙이 없어 단일 팔로우 팬(약 67%)만 대상으로 했다. 이번에는 **팔로우 중인 모든 아티스트의 공백일 중 최솟값(=가장 최근에 소통한 아티스트 기준)**을 그날의 "체감 공백"으로 정의해서, 활성 팔로우가 하나라도 있는 모든 (팬,날짜) 조합(100%)을 포함해 다시 계산한다. 최솟값을 쓰는 이유는 "여러 아티스트를 팔로우하는 팬은 그중 한 아티스트만 소통해도 계속 활동을 유지할 수 있다"는 논리를 그대로 반영하기 위해서다.


In [ ]:
all_follow_sql = """
WITH date_spine AS (
    SELECT DISTINCT activity_date_kst FROM mart_user_daily
),
active_follows AS (
    SELECT b.user_id, d.activity_date_kst, b.artist_id
    FROM bridge_user_artist_follow b
    CROSS JOIN date_spine d
    WHERE (b.followed_at_utc AT TIME ZONE 'Asia/Seoul')::date <= d.activity_date_kst
      AND (b.unfollowed_at_utc IS NULL OR (b.unfollowed_at_utc AT TIME ZONE 'Asia/Seoul')::date >= d.activity_date_kst)
),
follow_gap AS (
    SELECT af.user_id, af.activity_date_kst, af.artist_id, a.days_since_last_communication
    FROM active_follows af
    JOIN mart_artist_daily a ON a.artist_id = af.artist_id AND a.activity_date_kst = af.activity_date_kst
),
min_gap_per_user_date AS (
    SELECT user_id, activity_date_kst,
           MIN(days_since_last_communication) AS min_gap,
           COUNT(DISTINCT artist_id) AS n_followed
    FROM follow_gap
    GROUP BY user_id, activity_date_kst
)
SELECT u.user_id, u.activity_date_kst, u.activity_status, m.min_gap AS days_since_last_communication, m.n_followed
FROM mart_user_daily u
JOIN min_gap_per_user_date m ON m.user_id = u.user_id AND m.activity_date_kst = u.activity_date_kst;
"""
h02_all_df = pd.read_sql(all_follow_sql, engine)
print(f"활성 팔로우가 있는 전체 (팬,날짜) 조합 수: {len(h02_all_df)}건 "
      f"(단일 팔로우만이었던 이전 분석은 {len(h02_df)}건)")

h02_all_df["gap_bucket"] = pd.cut(h02_all_df["days_since_last_communication"], bins=bins, labels=labels)
bucket_summary_all = h02_all_df.groupby("gap_bucket", observed=True).agg(
    n=("user_id", "size"),
    at_risk_count=("activity_status", lambda s: (s == "at_risk").sum()),
    dormant_count=("activity_status", lambda s: (s == "dormant").sum()),
).reset_index()
bucket_summary_all["at_risk_pct"] = round(100 * bucket_summary_all["at_risk_count"] / bucket_summary_all["n"], 2)
bucket_summary_all["dormant_pct"] = round(100 * bucket_summary_all["dormant_count"] / bucket_summary_all["n"], 2)

comparison = bucket_summary[["gap_bucket", "n", "at_risk_pct", "dormant_pct"]].merge(
    bucket_summary_all[["gap_bucket", "n", "at_risk_pct", "dormant_pct"]],
    on="gap_bucket", suffixes=("_단일팔로우(67%)", "_전체팬(100%,최솟값기준)"),
)
comparison


## H-02 다중 팔로우 포함 재분석 결과 해석

- **결론(방향)이 그대로 유지된다.** 전체 팬(100%, 최솟값 공백 기준)에서도 `at_risk_pct`가 1.88% → 3.42% → 6.02% → 7.83%로, `dormant_pct`가 0.03% → 0.06% → 0.27% → 0.52%로 공백 구간이 길어질수록 뚜렷하게 단조 증가한다. 단일 팔로우만 봤을 때(2.44/3.71/6.24/7.83%, 0.04/0.08/0.34/0.52%)와 사실상 동일한 패턴이다.
- **예상했던 대로, 전체 팬 버전의 비율이 각 구간에서 단일 팔로우 버전보다 같거나 낮다**(예: "0일" 구간 1.88% vs 2.44%, "1-3일" 구간 3.42% vs 3.71%). 이는 다중 팔로우 팬이 팔로우 중인 다른 아티스트의 소통 덕분에 계속 활동을 유지해 평균을 낮추는 방향으로 작용한다는 가설과 일치한다. 다만 "8일 이상" 구간은 두 버전의 표본(n=1,532)이 완전히 같아 차이가 없는데, 이는 여러 아티스트를 동시에 팔로우하면서 모든 아티스트의 공백이 동시에 8일을 넘기는 경우 자체가 드물어(아티스트별 비활동기 시점이 서로 다르게 배치돼 있으므로) 이 구간에는 사실상 단일 팔로우 팬만 남았기 때문으로 보인다.
- **11.5절 결론**: 전체 팬(100%)을 포함해도 H-02 지지 방향은 바뀌지 않았고, 오히려 다중 팔로우 팬을 포함하면 비율이 전반적으로 낮아진다는(=다중 팔로우가 "보호 효과"를 준다는) 추가 근거까지 확인했다. 이제부터 이 "전체 팬(100%)" 버전을 H-02의 주 결과로 삼고, 단일 팔로우 버전은 대조군으로 남긴다.


## H-02 결과 요약과 해석 (절제된 표현)

- 위 차트는 단일 팔로우 팬(67%)만 대상으로 한 첫 번째 분석이다. **주 결론은 그 아래 "다중 팔로우 팬 포함 재분석"의 전체 팬(100%) 버전을 기준으로 삼는다** — 두 버전 모두 공백이 길수록 이탈위험률·휴면율이 뚜렷하게 단조 증가하는 같은 방향을 보였다.
- 표본 수(n)가 구간별로 크게 다르다(0일 구간 n=36,811~58,810 vs 8일 이상 구간 n=1,532). 구간 간 비율 차이가 크더라도 표본이 작은 구간(특히 "8일 이상")의 수치는 상대적으로 더 신중하게 다룬다.
- 다중 팔로우 팬을 포함하자 전체 팬 버전의 비율이 단일 팔로우 버전보다 같거나 낮게 나온 것은, 여러 아티스트를 팔로우하면 그중 하나가 소통을 쉬어도 다른 아티스트의 소통으로 계속 활동을 유지할 수 있다는 가설과 일치하는 방향이다.
- 상관관계·비율 비교이므로 인과관계로 해석하지 않는다(PRD 7.1절 H-02 주의점).


---
# 종합 결론

- **H-01**: 원본(동시) 상관은 아티스트를 합쳤을 때만 유의한 양의 상관을 보였고 개별 아티스트 단위에서는 방향이 엇갈렸다(특히 `artist_001`은 음의 상관). 그런데 누적 가입자 수(신규 가입 추세)를 통제한 편상관계수에서는 `artist_001`의 상관이 `-0.449`→`+0.580`으로 부호가 뒤집혔고 `artist_002`는 `0.429`→`0.670`(`p=0.024`, 유의)으로 강화됐다 — 즉 원본 상관의 엇갈림 상당 부분이 신규 가입자 유입이라는 교란 요인 때문이었을 가능성이 높다는 것이 확인됐다. 그럼에도 표본이 작아(`n=12`) `artist_002`를 제외하면 통계적으로 유의하지는 않으므로, H-01은 여전히 "가설과 같은 방향의 경향이 통제 후 더 뚜렷해졌으나, 개별 아티스트 수준에서 확정적으로 지지된다고 말하기는 이르다"는 절제된 결론으로 정리한다.
- **H-02**: 단일 팔로우 팬(67%)과 전체 팬(100%, 다중 팔로우 포함, 최솟값 공백 기준) 두 버전 모두에서 공백 구간이 길어질수록 14일 이탈위험률·30일 휴면율이 뚜렷하게 단조 증가했다(전체 팬 기준 at_risk 1.88%→3.42%→6.02%→7.83%, dormant 0.03%→0.06%→0.27%→0.52%). 전체 팬 버전이 단일 팔로우 버전보다 비율이 같거나 낮은 것은 다중 팔로우의 "보호 효과" 가설과도 일치한다. H-02는 지지되는 것으로 정리한다.
- **null_effect 재검증**: 1,750명 규모로 재생성한 null_effect 데이터에서 활동기-비활동기 노출률 차이가 `+0.0022`(사실상 0)로, baseline의 `+0.0276`과 뚜렷이 대비됐다. `docs/decisions_log.md` 5.11절(400명 규모)의 패턴이 최종 규모에서도 재현되어, 관찰된 소통-노출 관계가 `communication_effect` 스위치에 의한 것임을 재확인했다(PRD 13.4절 순환 논리 방지 장치 검증 통과).
- 두 가설 모두 컴백·공연 등 동시 영향, 표본 크기, 선택 편향(활동성 높은 팬이 원래 소통에 더 자주 노출) 가능성을 배제하지 못한다(PRD 2.4절 분석 제약).
- 이 노트북에서 발생한 해석적 판단(상관계수 강도 기준, 아티스트별 주간 참여율 정의, H-02 단일/다중 팔로우 처리, 편상관계수 통제 변수 선택)은 `docs/decisions_log.md` "11. 검토 필요 항목"에 기록한다.
